# Maffay — combine Q4 2017–2026 Q1 (digital + physical)

Reads every workbook in `as received 20260916` except the 2023 notes file. Q4 2017 line items are one former tab per `.xlsx` file; the 2017 summary workbook is skipped. Maps columns from **Translation tables.xlsx** (P–V), derives the red-font date fields (notes below row 28), then harmonises DSPs (`DSP.xlsx`), titles (`Title.xlsx`), countries (`Country.xlsx`), income type (`Income_type.xlsx`) and source (`Source.xlsx`).

Outputs go to `/Users/johannesnatterer/Developer/_output`:

- `Maffay_2017Q4_2026Q1_combined_harmonised.csv` — full combined table (too large for Excel’s row limit)
- `Maffay_2017Q4_2026Q1_combined_analysis.xlsx` — pivot written to the `data` sheet; other tabs are copied from the analysis workbook in `20260920. Detailed analysis 2017-2026` when that file exists
- `Maffay_2017Q4_2026Q1_filtered_titles.xlsx` — rows for the titles in `FILTER_TITLES` (written only if that list is not empty)
- `Maffay_2017Q4_2026Q1_combined_harmonised_first1000.xlsx` — first 1,000 rows
- `Maffay_2017Q4_2026Q1_control_check.xlsx` — combined Net Amount vs the amount column on each mapped source tab
- `Maffay_2017Q4_2026Q1_unmatched_DSP.xlsx` / `Maffay_2017Q4_2026Q1_unmatched_titles.xlsx` / `Maffay_2017Q4_2026Q1_unmatched_countries.xlsx` / `Maffay_2017Q4_2026Q1_unmatched_income_type.xlsx` / `Maffay_2017Q4_2026Q1_unmatched_source.xlsx` — written only if there are unmatched values

In the first code cell: `LOAD_COMBINED_CSV = True` reads the existing combined CSV instead of rebuilding from the quarterly workbooks; `FILTER_TITLES` is a comma-separated list of `Product Title (harmonised)` values to export. **Run All**, or run the last cells on their own after the CSV exists.


In [ ]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# Outputs: Maffay_2017Q4_2026Q1_*. Pivot is written into combined_analysis.xlsx sheet "data".

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

onedrive_root = Path.home() / "Library/CloudStorage/OneDrive-NattererLabs"
maffay_root = onedrive_root / "22 Fullstream Music/202607. Peter Maffay"
as_received = maffay_root / "Data/as received 20260916"
lookup_dir = maffay_root / "Data/lookup"
output_root = Path("/Users/johannesnatterer/Developer/_output")
output_root.mkdir(parents=True, exist_ok=True)

combined_csv = output_root / "Maffay_2017Q4_2026Q1_combined_harmonised.csv"
combined_first1000 = output_root / "Maffay_2017Q4_2026Q1_combined_harmonised_first1000.xlsx"
analysis_dir = maffay_root / "Data/Analyse JN/20260920. Detailed analysis 2017-2026"
analysis_filename = "Maffay_2017Q4_2026Q1_combined_analysis.xlsx"
analysis_template = analysis_dir / analysis_filename
analysis_output = output_root / analysis_filename
unmatched_dsp_xlsx = output_root / "Maffay_2017Q4_2026Q1_unmatched_DSP.xlsx"
unmatched_titles_xlsx = output_root / "Maffay_2017Q4_2026Q1_unmatched_titles.xlsx"
unmatched_countries_xlsx = output_root / "Maffay_2017Q4_2026Q1_unmatched_countries.xlsx"
unmatched_income_type_xlsx = output_root / "Maffay_2017Q4_2026Q1_unmatched_income_type.xlsx"
unmatched_source_xlsx = output_root / "Maffay_2017Q4_2026Q1_unmatched_source.xlsx"
check_xlsx = output_root / "Maffay_2017Q4_2026Q1_control_check.xlsx"
filtered_xlsx = output_root / "Maffay_2017Q4_2026Q1_filtered_titles.xlsx"

# a) True = load the existing combined CSV. False = rebuild from the quarterly workbooks.
LOAD_COMBINED_CSV = False

# b) Comma-separated Product Title (harmonised) values to export to xlsx. Leave empty to skip.
FILTER_TITLES = "Über sieben Brücken mußt du geh'n, So bist du, Sonne in der Nacht"   # e.g. "Tattoos - Live, Tabaluga"

# Set this to False if you do not want the source-file check to run.
RUN_SOURCE_CHECK = False

print("as received:      ", as_received)
print("lookup:           ", lookup_dir)
print("output:           ", output_root)
print("analysis folder:  ", analysis_dir)
print("analysis template:", analysis_template, "OK" if analysis_template.exists() else "(not found)")
print("analysis output:  ", analysis_output)
print("LOAD_COMBINED_CSV:", LOAD_COMBINED_CSV)
print("FILTER_TITLES:    ", FILTER_TITLES or "(none)")
print("RUN_SOURCE_CHECK: ", RUN_SOURCE_CHECK)

# Translation tables.xlsx column V (mapped list)
CANONICAL = [
    "Tab Name",
    "File Name",
    "Artist",
    "Product Title",
    "ISRC",
    "Reporting Quarter",
    "Reporting Month",
    "Sales Quarter",
    "Sales Month",
    "Distribution Channel",
    "Distribution Manner",
    "Third Party / DSP",
    "Third Party / DSP (Product)",
    "Net Invoice Price",
    "Received Rate %",
    "Amount per Unit",
    "SAP Sales Units",
    "Net Amount",
    "Country of Sale",
    "Product Configuration",
]

SKIP_FILE_PATTERNS = ("erläuterung", "erlauterung")
SKIP_SHEET_PATTERNS = (
    "rechnungsstellung",
    "analyse",
    "anaylse",
    "manufacturing",
    "hidden",
    "steuerliche",
    "m&p",
)


def skip_file(path):
    name = path.name.lower()
    if any(p in name for p in SKIP_FILE_PATTERNS):
        return True
    # Q4 2017 Red Rooster.xlsx is the summary workbook; line items live in the per-tab xlsx files.
    if name == "q4 2017 red rooster.xlsx":
        return True
    if path.suffix.lower() == ".xls" and path.with_suffix(".xlsx").exists():
        return True
    return False


def canonical_tab_name(path, sheet):
    """Q4 2017 stores each 2018-style tab as its own workbook."""
    name = path.name.lower()
    if "2017" not in name:
        return sheet
    is_mtv = "mtv" in name
    if "digital" in name:
        return "dig.GSA Red Rooster MTV Unpl." if is_mtv else "digital GSA Red Rooster"
    if "physical" in name:
        return "phys.GSA Red Rooster MTV Unpl." if is_mtv else "physisch GSA Red Rooster"
    return sheet


def skip_sheet(name):
    n = (name or "").lower()
    if n.startswith("xl_") or n.startswith("_com."):
        return True
    return any(p in n for p in SKIP_SHEET_PATTERNS)


def normalize_header(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    text = str(value).replace("\n", " ").replace("\r", " ").strip()
    return re.sub(r"\s+", " ", text)


def find_col(df, *candidates):
    lookup = {normalize_header(c).casefold(): c for c in df.columns}
    for candidate in candidates:
        key = normalize_header(candidate).casefold()
        if key in lookup:
            return lookup[key]
    return None


def series(df, *candidates):
    found = find_col(df, *candidates)
    if found is None:
        return pd.Series(pd.NA, index=df.index)
    return df[found]


def to_num(s):
    return pd.to_numeric(s, errors="coerce")


def to_year(s):
    return to_num(s)


def to_month(s):
    return to_num(s)


def yyyy_mm(year, month):
    """Format W/X example: 2025 12."""
    y = to_year(year)
    m = to_month(month)
    result = pd.Series(pd.NA, index=year.index, dtype="object")
    mask = y.notna() & m.notna() & (m >= 1) & (m <= 12)
    result.loc[mask] = (
        y.loc[mask].astype(int).astype(str)
        + " "
        + m.loc[mask].astype(int).astype(str).str.zfill(2)
    )
    return result


def month_to_quarter(month_series):
    """Turn 'YYYY MM' into 'YYYY QN' (translation table W/X)."""
    result = pd.Series(pd.NA, index=month_series.index, dtype="object")
    mask = month_series.notna()
    if not mask.any():
        return result
    parts = month_series.loc[mask].astype(str).str.split(" ", n=1, expand=True)
    if parts.shape[1] < 2:
        return result
    month_n = pd.to_numeric(parts[1], errors="coerce")
    ok = month_n.notna() & (month_n >= 1) & (month_n <= 12)
    if not ok.any():
        return result
    quarters = ((month_n.loc[ok].astype(int) - 1) // 3 + 1).astype(str)
    result.loc[parts.index[ok]] = parts.loc[ok, 0].astype(str) + " Q" + quarters
    return result


def parse_qn_token(value):
    """Parse 'Q1-2018', '2018 Q1', '1. Quartal 2018' → (year, 'Q1')."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return (np.nan, np.nan)
    text = str(value).strip()
    m = re.search(r"(20\d{2})\s*Q\s*([1-4])", text, flags=re.I)
    if m:
        return (int(m.group(1)), f"Q{m.group(2)}")
    m = re.search(r"Q\s*([1-4])\s*[-/ ]\s*(20\d{2})", text, flags=re.I)
    if m:
        return (int(m.group(2)), f"Q{m.group(1)}")
    m = re.search(r"([1-4])\.\s*Quartal\s*(20\d{2})", text, flags=re.I)
    if m:
        return (int(m.group(2)), f"Q{m.group(1)}")
    return (np.nan, np.nan)


def qn_yyyy_to_yyyy_qq(series_in):
    parsed = series_in.map(parse_qn_token)
    result = pd.Series(pd.NA, index=series_in.index, dtype="object")
    years = parsed.map(lambda t: t[0])
    qs = parsed.map(lambda t: t[1])
    mask = years.notna() & qs.notna()
    result.loc[mask] = years.loc[mask].astype(int).astype(str) + " " + qs.loc[mask].astype(str)
    already = series_in.astype(str).str.match(r"^\s*20\d{2}\s+Q[1-4]\s*$", na=False)
    result.loc[already & result.isna()] = series_in.loc[already & result.isna()].astype(str).str.strip()
    return result


def year_from_qn(series_in):
    parsed = series_in.map(parse_qn_token)
    years = parsed.map(lambda t: t[0])
    numeric = to_year(series_in)
    out = pd.to_numeric(years, errors="coerce")
    out = out.fillna(numeric)
    return out


NUMERIC_COLS = {
    "Net Invoice Price",
    "Received Rate %",
    "Amount per Unit",
    "SAP Sales Units",
    "Net Amount",
}


def empty_canonical(index):
    data = {}
    for column in CANONICAL:
        if column in NUMERIC_COLS:
            data[column] = pd.Series(np.nan, index=index, dtype="float64")
        else:
            data[column] = pd.Series(pd.NA, index=index, dtype="object")
    return pd.DataFrame(data)


def drop_trailing_summary(df):
    if df.empty:
        return df
    first_col = df.columns[0]
    last_valid = df[first_col].last_valid_index()
    if last_valid is None:
        return df.iloc[0:0]
    if last_valid < len(df) - 1:
        return df.loc[:last_valid]
    return df


def classify_sheet(df, sheet_name):
    """Map a sheet to translation-table groups P / Q / R / S / T / U."""
    if find_col(df, "WHOLESALE_VALUE"):
        return "P"
    if find_col(df, "Net Amount"):
        return "Q"
    sheet_l = (sheet_name or "").lower()
    is_phys = "phys" in sheet_l
    if find_col(df, "Abrechnungsmonat") or find_col(df, "Verkaufsland") or is_phys:
        if find_col(df, "Jahr") and not find_col(df, "Abrechnungsmonat"):
            return "S"
        if find_col(df, "Abrechnungsmonat"):
            sample = series(df, "Abrechnungsmonat").dropna().head(20)
            looks_qn = sample.astype(str).str.contains(r"Q\s*[1-4]", case=False, na=False).any()
            return "R" if looks_qn else "S"
        if is_phys and find_col(df, "Monat"):
            return "S"
    if find_col(df, "Zeitraum") and (find_col(df, "ISRC") or find_col(df, "Expr1003")):
        if find_col(df, "Expr1003") and not find_col(df, "ISRC"):
            return "U"
        return "T"
    return None


def map_group_p(df):
    """WHOLESALE_VALUE tabs (digital GSA from 2020 Q4)."""
    out = empty_canonical(df.index)
    out["Artist"] = series(df, "INTERPRET")
    out["Product Title"] = series(df, "TITEL")
    out["ISRC"] = series(df, "ISRC")
    reporting_month = yyyy_mm(series(df, "Jahr"), series(df, "Monat"))
    sales_month = yyyy_mm(series(df, "EMD_YR"), series(df, "EMD_PERIOD_ID"))
    out["Reporting Month"] = reporting_month
    out["Reporting Quarter"] = month_to_quarter(reporting_month)
    out["Sales Month"] = sales_month
    out["Sales Quarter"] = month_to_quarter(sales_month)
    out["Distribution Channel"] = series(df, "PARENT_SALES_CAT_DC")
    out["Distribution Manner"] = series(df, "SALES_CATEGORY_DC")
    out["Third Party / DSP"] = series(df, "PROVIDER_NAME")
    out["Third Party / DSP (Product)"] = series(df, "VENDOR_NAME")
    out["SAP Sales Units"] = to_num(series(df, "QUANTITY"))
    out["Net Amount"] = to_num(series(df, "WHOLESALE_VALUE"))
    out["Country of Sale"] = series(df, "COUNTRY_SALE")
    out["Product Configuration"] = series(df, "TONTRAEGER_BEZ")
    return out


def map_group_q(df):
    """Net Amount tabs (Ex-GSA, GVL, Kopplungserlöse)."""
    out = empty_canonical(df.index)
    out["Artist"] = series(df, "Product Main Artist")
    out["Product Title"] = series(df, "Product Title")
    out["ISRC"] = series(df, "ISRC")
    out["Reporting Quarter"] = qn_yyyy_to_yyyy_qq(series(df, "Accounting Period"))
    out["Reporting Month"] = series(df, "Reported Period")
    out["Sales Quarter"] = qn_yyyy_to_yyyy_qq(series(df, "Sales Period"))
    out["Distribution Channel"] = series(df, "Distribution Channel")
    out["Distribution Manner"] = series(df, "Distribution Manner")
    out["Third Party / DSP"] = series(df, "Third Party / DSP")
    out["Net Invoice Price"] = to_num(series(df, "Net Invoice Price"))
    out["Received Rate %"] = to_num(series(df, "Received Rate %"))
    out["Amount per Unit"] = to_num(series(df, "Amount per Unit"))
    out["SAP Sales Units"] = to_num(series(df, "SAP Sales Units"))
    out["Net Amount"] = to_num(series(df, "Net Amount"))
    out["Country of Sale"] = series(df, "Country of Sale")
    out["Product Configuration"] = series(df, "Product Configuration")
    return out


def physical_amount(df):
    """Net Amount for physical tabs: Translation tables R22/S22 = Nettoumsatz Vormonat."""
    return to_num(series(df, "Nettoumsatz Vormonat"))


def map_physical(df, reporting_month, reporting_quarter):
    out = empty_canonical(df.index)
    out["Artist"] = series(df, "Interpret")
    out["Product Title"] = series(df, "Titel")
    out["Reporting Quarter"] = reporting_quarter
    out["Reporting Month"] = reporting_month
    out["Distribution Channel"] = series(df, "Verkaufsweg")
    out["Amount per Unit"] = to_num(series(df, "HAP"))
    out["SAP Sales Units"] = to_num(series(df, "Nettoabsatz Vormonat"))
    out["Net Amount"] = physical_amount(df)
    out["Country of Sale"] = series(df, "Verkaufsland", "Tontr#")
    out["Product Configuration"] = series(df, "Raas Tontraeger Kat Bez", "Tontr#Kat#Bez")
    return out


def map_group_r(df):
    """Physical until ~2025 Q3: Abrechnungsmonat + Monat."""
    abr = series(df, "Abrechnungsmonat")
    year = year_from_qn(abr)
    month = to_month(series(df, "Monat"))
    reporting_month = yyyy_mm(year, month)
    reporting_quarter = qn_yyyy_to_yyyy_qq(abr)
    missing_q = reporting_quarter.isna() & reporting_month.notna()
    reporting_quarter.loc[missing_q] = month_to_quarter(reporting_month.loc[missing_q])
    return map_physical(df, reporting_month, reporting_quarter)


def map_group_s(df):
    """Physical from 2025 Q4: Jahr + Monat (or Abrechnungsmonat holding a year)."""
    year = to_year(series(df, "Jahr"))
    if year.isna().all():
        year = to_year(series(df, "Abrechnungsmonat"))
    month = to_month(series(df, "Monat"))
    reporting_month = yyyy_mm(year, month)
    reporting_quarter = month_to_quarter(reporting_month)
    return map_physical(df, reporting_month, reporting_quarter)


def map_group_tu(df):
    """Digital 2018–2020 Q3: Zeitraum + Monat. ISRC or Expr1003."""
    out = empty_canonical(df.index)
    out["Artist"] = series(df, "Interpret")
    out["Product Title"] = series(df, "Titel")
    out["ISRC"] = series(df, "ISRC", "Expr1003")
    zeitraum = series(df, "Zeitraum")
    year = year_from_qn(zeitraum)
    month = to_month(series(df, "Monat"))
    reporting_month = yyyy_mm(year, month)
    reporting_quarter = qn_yyyy_to_yyyy_qq(zeitraum)
    missing_q = reporting_quarter.isna() & reporting_month.notna()
    reporting_quarter.loc[missing_q] = month_to_quarter(reporting_month.loc[missing_q])
    out["Reporting Quarter"] = reporting_quarter
    out["Reporting Month"] = reporting_month
    out["Distribution Channel"] = series(df, "EMD_Parent_Sales_Cat_Bez")
    out["SAP Sales Units"] = to_num(series(df, "Nettoabs#"))
    out["Net Amount"] = to_num(series(df, "Netto-Ums#"))
    out["Country of Sale"] = series(df, "Land")
    out["Product Configuration"] = series(df, "Tontr#Kat#Bez")
    return out


MAPPERS = {
    "P": map_group_p,
    "Q": map_group_q,
    "R": map_group_r,
    "S": map_group_s,
    "T": map_group_tu,
    "U": map_group_tu,
}


def load_sheet(path, sheet):
    df = pd.read_excel(path, sheet_name=sheet)
    if df.empty:
        return df
    df.columns = [normalize_header(c) for c in df.columns]
    df = df.loc[:, ~pd.Index(df.columns).duplicated()]
    unnamed = [c for c in df.columns if not c or str(c).lower().startswith("unnamed")]
    if unnamed:
        df = df.drop(columns=unnamed, errors="ignore")
    df = drop_trailing_summary(df)
    df = df.dropna(how="all")
    return df.reset_index(drop=True)


def process_workbook(path):
    frames = []
    try:
        xl = pd.ExcelFile(path)
    except Exception as exc:
        print(f"  SKIP file (cannot open): {path.name} — {exc}")
        return frames
    for sheet in xl.sheet_names:
        if skip_sheet(sheet):
            continue
        try:
            df = load_sheet(path, sheet)
        except Exception as exc:
            print(f"  SKIP sheet {path.name} / {sheet!r}: {exc}")
            continue
        if df.empty or len(df.columns) < 4:
            continue
        group = classify_sheet(df, sheet)
        if group is None:
            looks_like_data = any(
                k in f"{sheet} {path.name}".lower()
                for k in ("phys", "dig", "gvl", "kopplung", "musical")
            )
            if looks_like_data:
                print(f"  UNMAPPED {path.name} / {sheet!r}  cols={list(df.columns)[:8]}")
            continue
        mapped = MAPPERS[group](df)
        mapped["Tab Name"] = canonical_tab_name(path, sheet)
        mapped["File Name"] = path.name
        mapped["Source Group"] = group
        frames.append(mapped)
        extra = ""
        if group in ("R", "S"):
            extra = f"  Net Amount={find_col(df, 'Nettoumsatz Vormonat')!r}"
        print(f"  {group}  {path.name} / {sheet!r}  → {len(mapped):,} rows{extra}")
    return frames


as received:       /Users/johannesnatterer/Library/CloudStorage/OneDrive-NattererLabs/22 Fullstream Music/202607. Peter Maffay/Data/as received 20260916
lookup:            /Users/johannesnatterer/Library/CloudStorage/OneDrive-NattererLabs/22 Fullstream Music/202607. Peter Maffay/Data/lookup
output:            /Users/johannesnatterer/Developer/_output
LOAD_COMBINED_CSV: False
FILTER_TITLES:     Über sieben Brücken mußt du geh'n, So bist du, Sonne in der Nacht
RUN_SOURCE_CHECK:  True


In [57]:
# Loading the raw input files is slow (large workbooks), so this only runs
# once per kernel session unless LOAD_COMBINED_CSV is True.
# Re-run this cell after a kernel restart, or if the input files themselves changed.
PHYSICAL_NET_AMOUNT_COL = "Nettoumsatz Vormonat"

if LOAD_COMBINED_CSV:
    if not combined_csv.exists():
        raise FileNotFoundError(
            f"Combined file not found: {combined_csv}\n"
            "Set LOAD_COMBINED_CSV = False in the first code cell to rebuild it."
        )
    combined = pd.read_csv(combined_csv, low_memory=False)
    LOADED_FROM_CSV = True
    print(f"Loaded {len(combined):,} rows from {combined_csv}")
    if "Source Group" in combined.columns:
        print(combined.groupby("Source Group").size().rename("rows").to_string())
    print(f"Net Amount total: {pd.to_numeric(combined['Net Amount'], errors='coerce').sum():,.2f}")
else:
    LOADED_FROM_CSV = False
    if "all_frames" in globals() and globals().get("LOADED_PHYSICAL_COL") != PHYSICAL_NET_AMOUNT_COL:
        print("Cached frames still used Summe for physical Net Amount; reloading from source files.")
        del all_frames
    if "all_frames" in globals() and not globals().get("LOADED_INCLUDE_2017"):
        print("Cached frames exclude Q4 2017; reloading from source files.")
        del all_frames

    if "all_frames" not in globals():
        files = sorted(
            p
            for p in as_received.iterdir()
            if p.suffix.lower() in {".xlsx", ".xls", ".xlsm"}
            and not p.name.startswith("~$")
            and not skip_file(p)
        )
        print(f"{len(files)} workbooks to process\n")

        all_frames = []
        for i, path in enumerate(files, 1):
            print(f"[{i}/{len(files)}] {path.name}")
            all_frames.extend(process_workbook(path))

        if not all_frames:
            raise RuntimeError("No sheets were mapped. Check as-received files and skip rules.")

        print("\nFinished loading input files.")
        LOADED_PHYSICAL_COL = PHYSICAL_NET_AMOUNT_COL
        LOADED_INCLUDE_2017 = True
    else:
        print("Input files already loaded in memory; skipping reload (re-run this cell if the input files changed).")

    combined = pd.concat(all_frames, ignore_index=True)
    print(f"Combined rows before lookup: {len(combined):,}")
    print(combined.groupby("Source Group").size().rename("rows").to_string())
    print(f"Net Amount total: {pd.to_numeric(combined['Net Amount'], errors='coerce').sum():,.2f}")


Input files already loaded in memory; skipping reload (re-run this cell if the input files changed).
Combined rows before lookup: 4,407,172
Source Group
P    2541463
Q    1714245
R       9922
S        381
T     107352
U      33809
Net Amount total: 12,516,574.51


In [58]:
if globals().get("LOADED_FROM_CSV"):
    print("Loaded from combined CSV; skipping harmonise.")
else:
    CODE_SUFFIX = re.compile(r"\s*\([A-Za-z]\|[A-Za-z0-9]+\|?\s*\)\s*$")
    CODE_INSIDE = re.compile(r"\(\s*([A-Za-z])\|([A-Za-z0-9]+)\|?\s*\)")


    def normalise_key(value):
        return re.sub(r"\s+", " ", str(value).strip()).casefold()


    def key_variants(value):
        """Match DSP/title keys across minor source-format differences."""
        raw = re.sub(r"\s+", " ", str(value).strip())
        variants = [raw, CODE_INSIDE.sub(r"(\1|\2| )", raw), CODE_INSIDE.sub(r"(\1|\2)", raw)]
        stripped = CODE_SUFFIX.sub("", raw).strip()
        if stripped and stripped != raw:
            variants.append(stripped)
        seen = set()
        out = []
        for variant in variants:
            key = normalise_key(variant)
            if key and key not in seen:
                seen.add(key)
                out.append(key)
        return out


    def build_lookup(lookup_map):
        normalised = {}
        for key, value in lookup_map.items():
            if pd.isna(key):
                continue
            for variant in key_variants(key):
                normalised.setdefault(variant, value)
        return normalised


    def harmonise(df, column, new_column, lookup_map, label, warn=True):
        normalised_lookup = build_lookup(lookup_map)

        def map_one(value):
            if pd.isna(value):
                return pd.NA
            for variant in key_variants(value):
                if variant in normalised_lookup:
                    return normalised_lookup[variant]
            return pd.NA

        mapped = df[column].map(map_one)
        missing_mask = df[column].notna() & mapped.isna()
        if warn and missing_mask.any():
            counts = df.loc[missing_mask, column].value_counts()
            print(
                f"WARNING: {len(counts)} '{column}' value(s) could not be harmonised "
                f"against {label} ({missing_mask.sum():,} rows):"
            )
            for value, count in counts.head(40).items():
                print(f"  {value!r} ({count:,} rows)")
            if len(counts) > 40:
                print(f"  … {len(counts) - 40} more")
            print()
        df[new_column] = mapped
        return df


    title_lookup_df = pd.read_excel(lookup_dir / "Title.xlsx")
    title_lookup = dict(zip(title_lookup_df["Product Title"], title_lookup_df["Product Title (harmonised)"]))
    dsp_lookup_df = pd.read_excel(lookup_dir / "DSP.xlsx")
    dsp_lookup = dict(zip(dsp_lookup_df["Third Party / DSP"], dsp_lookup_df["Third Party / DSP (Group)"]))
    country_lookup_df = pd.read_excel(lookup_dir / "Country.xlsx")
    country_lookup = dict(zip(country_lookup_df["Country of Sale"], country_lookup_df["Country of Sale (harmonised)"]))
    gsa_lookup = dict(zip(country_lookup_df["Country of Sale"], country_lookup_df["GSA"]))
    income_type_lookup_df = pd.read_excel(lookup_dir / "Income_type.xlsx")
    income_type_lookup = dict(zip(income_type_lookup_df["Tab Name"], income_type_lookup_df["Type of income"]))
    source_lookup_df = pd.read_excel(lookup_dir / "Source.xlsx")
    source_lookup = dict(zip(source_lookup_df["Tab Name"], source_lookup_df["Source"]))

    combined = harmonise(combined, "Product Title", "Product Title (harmonised)", title_lookup, "Title.xlsx")
    combined = harmonise(combined, "Third Party / DSP", "Third Party / DSP (Group)", dsp_lookup, "DSP.xlsx")
    combined = harmonise(combined, "Country of Sale", "Country of Sale (harmonised)", country_lookup, "Country.xlsx")
    combined = harmonise(combined, "Country of Sale", "GSA", gsa_lookup, "Country.xlsx", warn=False)
    combined = harmonise(combined, "Tab Name", "Type of income", income_type_lookup, "Income_type.xlsx")
    combined = harmonise(combined, "Tab Name", "Source", source_lookup, "Source.xlsx")

    final_columns = []
    for column in CANONICAL:
        final_columns.append(column)
        if column == "Tab Name":
            final_columns.append("Type of income")
            final_columns.append("Source")
        if column == "Product Title":
            final_columns.append("Product Title (harmonised)")
        if column == "Third Party / DSP":
            final_columns.append("Third Party / DSP (Group)")
        if column == "Country of Sale":
            final_columns.append("Country of Sale (harmonised)")
            final_columns.append("GSA")
    final_columns.append("Source Group")
    combined = combined.reindex(columns=final_columns)
    combined.head(3)


In [59]:
if globals().get("LOADED_FROM_CSV"):
    print("Loaded from combined CSV; not rewriting the main CSV.")
else:
    combined.to_csv(combined_csv, index=False)
    print(f"Wrote {len(combined):,} rows → {combined_csv}")

    sample = combined.head(1000)
    sample.to_excel(combined_first1000, index=False)
    print(f"Wrote first {len(sample):,} rows → {combined_first1000}")


    def unmatched_counts(df, source_col, mapped_col, label):
        missing = df[source_col].notna() & df[mapped_col].isna()
        counts = (
            df.loc[missing, source_col]
            .value_counts()
            .rename_axis(source_col)
            .reset_index(name="Rows")
        )
        print(f"{len(counts)} unmatched {label} ({missing.sum():,} rows)")
        return counts


    unmatched_dsp = unmatched_counts(combined, "Third Party / DSP", "Third Party / DSP (Group)", "DSPs")
    unmatched_titles = unmatched_counts(combined, "Product Title", "Product Title (harmonised)", "titles")
    unmatched_countries = unmatched_counts(combined, "Country of Sale", "Country of Sale (harmonised)", "countries")
    unmatched_income_type = unmatched_counts(combined, "Tab Name", "Type of income", "income types")
    unmatched_source = unmatched_counts(combined, "Tab Name", "Source", "sources")


    def write_unmatched(counts, path, label):
        if counts.empty:
            print(f"No unmatched {label}; not writing {path.name}")
            if path.exists():
                path.unlink()
            return
        counts.to_excel(path, index=False)
        print(f"Wrote unmatched {label} → {path}")


    write_unmatched(unmatched_dsp, unmatched_dsp_xlsx, "DSPs")
    write_unmatched(unmatched_titles, unmatched_titles_xlsx, "titles")
    write_unmatched(unmatched_countries, unmatched_countries_xlsx, "countries")
    write_unmatched(unmatched_income_type, unmatched_income_type_xlsx, "income types")
    write_unmatched(unmatched_source, unmatched_source_xlsx, "sources")


Wrote 4,407,172 rows → /Users/johannesnatterer/Developer/_output/Maffay_2018_2026Q1_combined_harmonised.csv
Wrote first 1,000 rows → /Users/johannesnatterer/Developer/_output/Maffay_2018_2026Q1_combined_harmonised_first1000.xlsx
0 unmatched DSPs (0 rows)
0 unmatched titles (0 rows)
0 unmatched countries (0 rows)
0 unmatched income types (0 rows)
0 unmatched sources (0 rows)
No unmatched DSPs; not writing Maffay_2018_2026Q1_unmatched_DSP.xlsx
No unmatched titles; not writing Maffay_2018_2026Q1_unmatched_titles.xlsx
No unmatched countries; not writing Maffay_2018_2026Q1_unmatched_countries.xlsx
No unmatched income types; not writing Maffay_2018_2026Q1_unmatched_income_type.xlsx
No unmatched sources; not writing Maffay_2018_2026Q1_unmatched_source.xlsx


In [60]:
# Filter the combined file by Product Title (harmonised) and write xlsx.
# Set FILTER_TITLES in the first code cell, comma-separated, for example:
# FILTER_TITLES = "Tattoos - Live, Tabaluga"

titles = [t.strip() for t in str(FILTER_TITLES).split(",") if t.strip()]
if not titles:
    print("FILTER_TITLES is empty; not writing a filtered file.")
else:
    if "combined" in globals():
        filter_source = combined
    else:
        filter_source = pd.read_csv(combined_csv, low_memory=False)
        print(f"Loaded {len(filter_source):,} rows from {combined_csv}")

    col = "Product Title (harmonised)"
    if col not in filter_source.columns:
        raise KeyError(f"Column {col!r} is not in the combined file.")

    wanted = set(titles)
    filtered = filter_source[filter_source[col].isin(wanted)].copy()
    found = set(filtered[col].dropna().unique())
    missing = sorted(wanted - found)

    print(f"Filter titles ({len(wanted)}): {', '.join(titles)}")
    print(f"Matched rows: {len(filtered):,}")
    if len(filtered):
        print(filtered.groupby(col).size().rename("rows").to_string())
        net = pd.to_numeric(filtered["Net Amount"], errors="coerce").sum()
        print(f"Net Amount: {net:,.2f}")
    if missing:
        print("Not found in Product Title (harmonised):")
        for title in missing:
            print(f"  {title!r}")

    excel_max = 1_048_575
    if filtered.empty:
        print("No matching rows; not writing a file.")
    elif len(filtered) > excel_max:
        out = output_root / "Maffay_2017Q4_2026Q1_filtered_titles.csv"
        filtered.to_csv(out, index=False)
        print(f"Too many rows for Excel; wrote CSV → {out}")
    else:
        filtered.to_excel(filtered_xlsx, index=False)
        print(f"Wrote {len(filtered):,} rows → {filtered_xlsx}")


Filter titles (3): Über sieben Brücken mußt du geh'n, So bist du, Sonne in der Nacht
Matched rows: 385,845
Product Title (harmonised)
So bist du                           100240
Sonne in der Nacht                   116376
Über sieben Brücken mußt du geh'n    169229
Net Amount: 986,223.17
Wrote 385,845 rows → /Users/johannesnatterer/Developer/_output/Maffay_2018_2026Q1_filtered_titles.xlsx


In [64]:
# Pivot only — run this cell after the combined CSV exists (or after the cells above).
# Change the index names (and optional columns) here; no need to re-run the combine.
pivot_index_1 = "Third Party / DSP (Group)" # None" # "Type of income"
pivot_index_2 = "Type of income" #"Third Party / DSP (Group)"  # set to None to drop this index
pivot_index_3 = None  # e.g. "Sales Month", or None to drop this index
pivot_index_4 = None # "Reporting Quarter" #"Third Party / DSP (Group)"  # set to None to drop this index
pivot_index_5 = None # "Sales Quarter" #"Third Party / DSP (Group)"  # set to None to drop this index
pivot_index_6 = "Product Title (harmonised)"  #"Third Party / DSP (Group)"  # set to None to drop this index
pivot_columns = None  # e.g. "Reporting Quarter", or None for a single total column

if "combined" in globals():
    pivot_source = combined
else:
    pivot_source = pd.read_csv(combined_csv, low_memory=False)
    print(f"Loaded {len(pivot_source):,} rows from {combined_csv}")

pivot_index = [name for name in [pivot_index_1, pivot_index_2, pivot_index_3, pivot_index_4, pivot_index_5, pivot_index_6] if name]
pivot_source = pivot_source.copy()


def fill_blank_category(series):
    text = series.astype("string").str.strip()
    missing = text.isna() | (text == "") | text.str.casefold().isin({"nan", "<na>", "none"})
    return text.mask(missing, "Blank")


for name in pivot_index + ([pivot_columns] if pivot_columns else []):
    if name in pivot_source.columns:
        empty = pivot_source[name].isna().sum()
        if empty:
            print(f"Filling {empty:,} empty values in '{name}' with 'Blank'")
        pivot_source[name] = fill_blank_category(pivot_source[name])

pivot = pd.pivot_table(
    pivot_source,
    values=["Net Amount", "SAP Sales Units"],
    index=pivot_index,
    columns=pivot_columns,
    fill_value=0,
    margins=False,
    aggfunc="sum",
)
pivot = pivot.reset_index()
if isinstance(pivot.columns, pd.MultiIndex):
    pivot.columns = [
        " ".join(str(part) for part in col if str(part) not in {"", "None", "nan"}).strip()
        if isinstance(col, tuple) else col
        for col in pivot.columns.to_list()
    ]
pivot[pivot_index] = pivot[pivot_index].replace("", pd.NA).ffill()

pivot.to_excel(pivot_xlsx, index=False, merge_cells=False)
print(f"Wrote pivot → {pivot_xlsx}")
print(
    "Pivot table - 'Net Amount' by "
    + " / ".join(pivot_index)
    + (" / " + pivot_columns if pivot_columns else "")
    + ":"
)
pivot

Filling 246,021 empty values in 'Third Party / DSP (Group)' with 'Blank'
Wrote pivot → /Users/johannesnatterer/Developer/_output/Maffay_2018_2026Q1_combined_pivot.xlsx
Pivot table - 'Net Amount' by Third Party / DSP (Group) / Type of income / Product Title (harmonised):


,Third Party / DSP (Group),Type of income,Product Title (harmonised),Net Amount,SAP Sales Units
0,7 Digital (global deal) (D|PJR5| ),Digital,"""Sonne in der Nacht"" Audiothek Interview",0.047317,3.0
1,7 Digital (global deal) (D|PJR5| ),Digital,Alles nur Kino,0.015772,1.0
2,7 Digital (global deal) (D|PJR5| ),Digital,Alter Mann,4.111175,12.0
3,7 Digital (global deal) (D|PJR5| ),Digital,Andy - Träume sterben jung,0.260000,1.0
4,7 Digital (global deal) (D|PJR5| ),Digital,Angela,0.260000,1.0
...,...,...,...,...,...
29385,iMusica LLC USA (D|PFS7| ),Digital,Über sieben Brücken mußt du geh'n,0.030000,2.0
29386,mora qualitas (D|PQZ4),Digital,Dreams on Fire,0.020000,1.0
29387,mora qualitas (D|PQZ4),Digital,Ich wollte nie erwachsen sein,0.020000,1.0
29388,"sms.at, Austria",Digital,Gelobtes Land,1.000000,1.0


In [ ]:
if not RUN_SOURCE_CHECK:
    print("Source-file check skipped (RUN_SOURCE_CHECK = False).")
else:
    # Source-file check: combined Net Amount vs the amount column on each mapped tab.
    # Trailing sheet totals (empty first column, amount = sum of lines) are dropped
    # the same way as in load_sheet. Run after the setup cell.
    # Uses `combined` if loaded, otherwise the CSV.

    TOLERANCE = 0.01  # EUR
    # Filename fragment to limit the run, or None for every as-received file.
    CHECK_FILE_FILTER = None


    def source_amount_series(df, group):
        if group == "P":
            return to_num(series(df, "WHOLESALE_VALUE"))
        if group == "Q":
            return to_num(series(df, "Net Amount"))
        if group in ("R", "S"):
            return physical_amount(df)
        if group in ("T", "U"):
            return to_num(series(df, "Netto-Ums#"))
        return pd.Series(dtype="float64")


    def source_amount_col(df, group):
        if group == "P":
            return find_col(df, "WHOLESALE_VALUE")
        if group == "Q":
            return find_col(df, "Net Amount")
        if group in ("R", "S"):
            return find_col(df, "Nettoumsatz Vormonat")
        if group in ("T", "U"):
            return find_col(df, "Netto-Ums#")
        return None


    if "combined" in globals():
        check_source = combined
    else:
        check_source = pd.read_csv(combined_csv, low_memory=False)
        print(f"Loaded {len(check_source):,} rows from {combined_csv}")

    check_source = check_source.copy()
    check_source["Net Amount"] = pd.to_numeric(check_source["Net Amount"], errors="coerce")
    combined_tot = (
        check_source.groupby(["File Name", "Tab Name"], dropna=False)["Net Amount"]
        .agg(combined_sum="sum", combined_n="count")
        .reset_index()
    )

    files = sorted(
        p
        for p in as_received.iterdir()
        if p.suffix.lower() in {".xlsx", ".xls", ".xlsm"}
        and not p.name.startswith("~$")
        and not skip_file(p)
    )
    if CHECK_FILE_FILTER:
        files = [p for p in files if CHECK_FILE_FILTER.lower() in p.name.lower()]
    print(f"{len(files)} workbooks to check against the combined file\n")

    rows = []
    for i, path in enumerate(files, 1):
        print(f"[{i}/{len(files)}] {path.name}", flush=True)
        try:
            xl = pd.ExcelFile(path)
        except Exception as exc:
            print(f"  CANNOT OPEN: {exc}")
            rows.append(
                {
                    "File Name": path.name,
                    "Tab Name": "(file)",
                    "status": "CANNOT_OPEN",
                    "note": str(exc),
                }
            )
            continue
        seen = set()
        for sheet in xl.sheet_names:
            if skip_sheet(sheet):
                continue
            try:
                df = load_sheet(path, sheet)
            except Exception as exc:
                print(f"  SKIP {sheet!r}: {exc}")
                continue
            if df.empty or len(df.columns) < 4:
                continue
            group = classify_sheet(df, sheet)
            if group is None:
                looks_like_data = any(
                    k in f"{sheet} {path.name}".lower()
                    for k in ("phys", "dig", "gvl", "kopplung", "musical")
                )
                if looks_like_data:
                    print(f"  UNMAPPED {path.name} / {sheet!r}  cols={list(df.columns)[:8]}")
                continue
            amount = source_amount_series(df, group)
            source_sum = float(amount.sum(skipna=True))
            source_n = int(len(df))
            tab_name = canonical_tab_name(path, sheet)
            match = combined_tot[
                (combined_tot["File Name"] == path.name) & (combined_tot["Tab Name"] == tab_name)
            ]
            combined_sum = float(match["combined_sum"].sum()) if len(match) else 0.0
            combined_n = int(match["combined_n"].sum()) if len(match) else 0
            diff = combined_sum - source_sum
            if abs(diff) <= TOLERANCE and combined_n == source_n:
                status = "OK"
            elif abs(diff) <= TOLERANCE:
                status = "SUM_OK"
            else:
                status = "MISMATCH"
            if status != "OK":
                print(
                    f"  {status} {sheet!r}  source={source_sum:,.2f} n={source_n}  "
                    f"combined={combined_sum:,.2f} n={combined_n}  diff={diff:,.4f}"
                )
            rows.append(
                {
                    "File Name": path.name,
                    "Tab Name": tab_name,
                    "Source Group": group,
                    "Amount column": source_amount_col(df, group),
                    "source_n": source_n,
                    "combined_n": combined_n,
                    "source_sum": source_sum,
                    "combined_sum": combined_sum,
                    "Difference": diff,
                    "status": status,
                }
            )
            seen.add(tab_name)
        extra = set(combined_tot.loc[combined_tot["File Name"] == path.name, "Tab Name"]) - seen
        for tab in sorted(extra):
            extra_sum = float(
                combined_tot.loc[
                    (combined_tot["File Name"] == path.name) & (combined_tot["Tab Name"] == tab),
                    "combined_sum",
                ].sum()
            )
            print(f"  EXTRA IN COMBINED {tab!r}  combined={extra_sum:,.2f}")
            rows.append(
                {
                    "File Name": path.name,
                    "Tab Name": tab,
                    "combined_sum": extra_sum,
                    "status": "EXTRA_IN_COMBINED",
                }
            )

    detail = pd.DataFrame(rows)
    by_file = (
        detail.groupby("File Name", dropna=False)[["source_sum", "combined_sum", "source_n", "combined_n"]]
        .sum(min_count=1)
        .reset_index()
    )
    by_file["Difference"] = by_file["combined_sum"] - by_file["source_sum"]
    by_file["Match"] = by_file["Difference"].abs() <= TOLERANCE

    n_ok = int((detail["status"] == "OK").sum()) if len(detail) else 0
    n_bad = int((detail["status"] != "OK").sum()) if len(detail) else 0

    print("\n===== All files =====")
    print(by_file.to_string(index=False))
    print(f"\nTabs OK: {n_ok} / {len(detail)}   (not OK: {n_bad})")
    print(f"Source tabs total:    {detail['source_sum'].sum():,.2f}")
    print(f"Combined tabs total:  {detail['combined_sum'].sum():,.2f}")
    print(f"Combined file total:  {check_source['Net Amount'].sum():,.2f}")

    mismatches = detail[detail["status"] != "OK"]
    if len(mismatches):
        print("\nTabs that did not match:")
        print(
            mismatches[
                [
                    "File Name",
                    "Tab Name",
                    "source_n",
                    "combined_n",
                    "source_sum",
                    "combined_sum",
                    "Difference",
                    "status",
                ]
            ].to_string(index=False)
        )
    else:
        print("\nAll mapped source tabs match the combined file (sum and row count).")

    with pd.ExcelWriter(check_xlsx, engine="openpyxl") as writer:
        by_file.to_excel(writer, sheet_name="By file", index=False)
        detail.to_excel(writer, sheet_name="By tab", index=False)
        if len(mismatches):
            mismatches.to_excel(writer, sheet_name="Mismatches", index=False)

    print(f"\nWrote check → {check_xlsx}")
    by_file